# 📊 Sales Data Analysis & Business Intelligence
### End-to-End Exploratory Data Analysis (EDA) & Portfolio Project

**Author:** Data Analytics Portfolio Project  
**Dataset:** 10,000+ Retail & E-Commerce Sales Transactions  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn  

---
### 🎯 Business Objectives
1. **Data Integrity:** Clean raw transaction logs, handle missing records, and eliminate duplicate entries.
2. **Sales Trends:** Uncover monthly and yearly growth patterns and seasonal peaks.
3. **Category & Product Performance:** Identify top revenue drivers and profit margins across categories.
4. **Geographic Distribution:** Pinpoint highest-performing regions and strategic growth areas.
5. **Customer & Payment Insights:** Understand customer ordering habits and preferred payment methods.

## 1. Environment Setup & Library Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('Libraries imported successfully!')

## 2. Load the Dataset
We load the sales dataset from the `../data/` directory.

In [ ]:
# Path to dataset
data_path = os.path.join('..', 'data', 'sales_data.csv')

df = pd.read_csv(data_path)
print(f'Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns')
df.head()

## 3. Data Inspection & Cleaning
- Inspect data types and null counts
- Verify and remove duplicate rows
- Convert date columns to appropriate formats

In [ ]:
# Dataset Summary
df.info()

# Check for Missing Values
missing = df.isnull().sum()
print('\nMissing Values Per Column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found!')

# Check for Duplicate Records
duplicates = df.duplicated().sum()
print(f'\nDuplicate rows found: {duplicates}')
if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print(f'Duplicates removed. Remaining rows: {len(df):,}')

## 4. Feature Engineering
Extract temporal features from `Order Date` and calculate key business metrics like `Profit Margin %`.

In [ ]:
# Parse dates
df['Order Date'] = pd.to_datetime(df['Order Date'])

# Extract time components
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Year_Month'] = df['Order Date'].dt.strftime('%Y-%m')
df['Month_Name'] = df['Order Date'].dt.strftime('%b')
df['Quarter'] = 'Q' + df['Order Date'].dt.quarter.astype(str)

# Compute Profit Margin %
df['Profit Margin %'] = (df['Profit'] / df['Sales'] * 100).round(2)

df[['Order ID', 'Order Date', 'Sales', 'Cost', 'Profit', 'Profit Margin %', 'Year_Month']].head()

## 5. Executive KPI Summary
High-level scorecard of commercial health.

In [ ]:
total_rev = df['Sales'].sum()
total_cst = df['Cost'].sum()
total_prf = df['Profit'].sum()
tot_orders = df['Order ID'].nunique()
tot_units = df['Quantity'].sum()
tot_custs = df['Customer ID'].nunique()
avg_margin = (total_prf / total_rev * 100)
aov = total_rev / tot_orders

print('='*50)
print('           EXECUTIVE KPI SCORECARD')
print('='*50)
print(f'Total Revenue:           ${total_rev:,.2f}')
print(f'Total Cost:              ${total_cst:,.2f}')
print(f'Total Net Profit:        ${total_prf:,.2f}')
print(f'Overall Profit Margin:   {avg_margin:.2f}%')
print(f'Total Orders:            {tot_orders:,}')
print(f'Total Units Sold:        {tot_units:,}')
print(f'Average Order Value:     ${aov:,.2f}')
print(f'Unique Active Customers: {tot_custs:,}')
print('='*50)

## 6. Monthly & Yearly Sales Trend Analysis
Analyzing sales trajectory and revenue patterns across months.

In [ ]:
monthly = df.groupby('Year_Month').agg(
    Revenue=('Sales', 'sum'),
    Profit=('Profit', 'sum')
).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 6))

# Revenue Bar Chart
ax1.bar(monthly['Year_Month'], monthly['Revenue'], color='#3b82f6', alpha=0.7, label='Monthly Revenue ($)')
ax1.set_ylabel('Revenue ($)', color='#1d4ed8', fontsize=12, fontweight='bold')
ax1.tick_params(axis='y', labelcolor='#1d4ed8')
ax1.set_xticklabels(monthly['Year_Month'], rotation=45, ha='right')

# Profit Line on secondary axis
ax2 = ax1.twinx()
ax2.plot(monthly['Year_Month'], monthly['Profit'], color='#10b981', marker='o', linewidth=2.5, label='Net Profit ($)')
ax2.set_ylabel('Net Profit ($)', color='#047857', fontsize=12, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='#047857')
ax2.grid(False)

plt.title('Monthly Sales Revenue & Net Profit Trend', fontsize=15, fontweight='bold', pad=15)
fig.tight_layout()
plt.show()

## 7. Category-Wise Performance Analysis
Which product categories generate the highest revenue and best profit margins?

In [ ]:
cat_summary = df.groupby('Category').agg(
    Revenue=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Units_Sold=('Quantity', 'sum')
).reset_index()
cat_summary['Profit_Margin_%'] = (cat_summary['Profit'] / cat_summary['Revenue'] * 100).round(2)
cat_summary.sort_values(by='Revenue', ascending=False, inplace=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Revenue share donut chart
axes[0].pie(
    cat_summary['Revenue'], 
    labels=cat_summary['Category'], 
    autopct='%1.1f%%', 
    startangle=140, 
    colors=sns.color_palette('pastel'),
    wedgeprops=dict(width=0.4, edgecolor='white')
)
axes[0].set_title('Revenue Share by Category', fontsize=14, fontweight='bold')

# Profit Margin Bar Chart
sns.barplot(data=cat_summary, x='Profit_Margin_%', y='Category', ax=axes[1], palette='crest')
axes[1].set_title('Profit Margin % by Category', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Profit Margin (%)')
for i, v in enumerate(cat_summary['Profit_Margin_%']):
    axes[1].text(v + 0.3, i, f'{v:.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Regional Commercial Performance
Comparison of sales, profit, and volume across business territories.

In [ ]:
region_summary = df.groupby('Region').agg(
    Revenue=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Orders=('Order ID', 'nunique')
).reset_index().sort_values(by='Revenue', ascending=False)

plt.figure(figsize=(12, 6))
ax = sns.barplot(data=region_summary, x='Region', y='Revenue', palette='Blues_r')
plt.title('Total Revenue by Geographic Region', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Revenue ($)', fontsize=12)

# Add data labels
for p in ax.patches:
    ax.annotate(f'${p.get_height():,.0f}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 5),
                textcoords='offset points')

plt.tight_layout()
plt.show()

## 9. Top 10 Best-Selling Products
Identifying the flagship items driving our top-line revenue.

In [ ]:
top_products = df.groupby(['Product Name', 'Category']).agg(
    Revenue=('Sales', 'sum'),
    Units_Sold=('Quantity', 'sum'),
    Profit=('Profit', 'sum')
).reset_index().sort_values(by='Revenue', ascending=False).head(10)

plt.figure(figsize=(14, 7))
ax = sns.barplot(data=top_products, y='Product Name', x='Revenue', hue='Category', dodge=False, palette='mako')
plt.title('Top 10 Products by Total Revenue', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Total Revenue ($)', fontsize=12)
plt.ylabel('Product Name', fontsize=12)

for p in ax.patches:
    width = p.get_width()
    if width > 0:
        ax.annotate(f'${width:,.0f}',
                    (width, p.get_y() + p.get_height() / 2.),
                    ha='left', va='center', fontsize=9, xytext=(5, 0),
                    textcoords='offset points')

plt.tight_layout()
plt.show()

## 10. Profit vs Sales Relationship
Analyzing margins and checking for potential loss-making orders or volume discounts.

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=df,
    x='Sales',
    y='Profit',
    hue='Category',
    size='Quantity',
    sizes=(20, 200),
    alpha=0.6,
    palette='Set2'
)
plt.title('Sales vs. Profit Correlation by Product Category', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Order Sales ($)', fontsize=12)
plt.ylabel('Order Profit ($)', fontsize=12)
plt.axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.7, label='Break-Even Line')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 11. Payment Mode & Customer Analysis
Preferred customer checkout options and top repeat buyers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Payment Mode Breakdown
pay_counts = df['Payment Mode'].value_counts()
axes[0].pie(pay_counts.values, labels=pay_counts.index, autopct='%1.1f%%', colors=sns.color_palette('pastel'), startangle=90)
axes[0].set_title('Payment Mode Preferences', fontsize=14, fontweight='bold')

# Top 10 Customers by Revenue
top_cust = df.groupby('Customer ID')['Sales'].sum().reset_index().sort_values(by='Sales', ascending=False).head(10)
sns.barplot(data=top_cust, x='Sales', y='Customer ID', ax=axes[1], palette='flare')
axes[1].set_title('Top 10 High-Value Customers', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Total Spend ($)')

plt.tight_layout()
plt.show()

## 12. Strategic Insights & Business Recommendations

1. **Technology Dominates Top-Line:** High-ticket items like Laptops and Smartphones contribute the majority of gross sales, while maintaining healthy ~25-30% margins.
2. **Office Supplies High Margin Stability:** Although lower in unit price, Office Supplies demonstrate high re-order volume and resilient profit margins (>35%).
3. **Seasonal Q4 Surge:** Monthly tracking indicates significant revenue spikes during Q4 (holiday shopping and year-end corporate budget clearing).
4. **Digital Payments Prevalent:** Credit Card and UPI/Debit transactions account for over 80% of revenue, indicating seamless checkout friction is paramount.